# NLP Preprocessing Engine (Advanced)

## Task 1: Conceptual Understanding.


### 1. What is the difference between "Love" and "love" in NLP?

- NLP models treat 'Love' and 'love' in a different way and see both as different models.
- Without lowercasing it doubles the vocabulary and might create noise data.

Example: **'Love is important'**, **'love is great'** --> here model sees 2 different words. by this sentiment analysis can miscount frequency which produces weaker predictions.

### 2. If stopwords are not removed


- If stopwords are not removed it creats a noisy heavy input.
- By this important words might get diluted.

Example:
- **`"This is not good"`**:  Here `is` and `this` add no value.
- **`"He went to the store"`**: Here `to` and `the` create more frequency and migt mislead the model


### 3. Provide two real-world scenarios where removing stopwords can be harmful.

2 real-world examples where removing stopwords can be harmful:

- 1. **Sentiment Analysis:** Reviews like “The product is not good” vs “The product is good” rely on words such as “not” to flip the polarity; if you remove “not” as a stopword, both reduce to “product good,” and the negative review will be misclassified as positive

- 2. **Query: “Who is the president of India in 2026?”**

    - After aggressive stopword removal (dropping “who”, “is”, “the”, “in”): “president India 2026” – the system may treat this like a generic fact lookup about the presidency instead of a current entity question, which can break QA models or retrieval that rely on the full question structure.



### 4. Explain the difference between stemming and lemmatization.

Stemming crudely chops word endings to get a root-like form, while lemmatization uses vocabulary and grammar to return a correct dictionary base form (lemma).

**Core difference**

- **Stemming**: Rule-based suffix stripping, often ignores part-of-speech and can produce non-words, e.g., “studies”, “studying” → “studi”.

- **Lemmatization**: Uses a lexicon and POS (and sometimes context) to get valid words, e.g., “studies” (verb), “studying” → “study”; “better” → “good”.

**Practical trade-off**
- Stemming is faster and simpler but less accurate and more aggressive; often used in search/IR where exact form is less important.

- Lemmatization is slower and more resource-intensive but more accurate and linguistically meaningful; preferred in tasks like text classification, QA, and chatbots where semantics matter.



## Task 2:Build Advanced Preprocessing Function


In [1]:
import re

def normalize_repeated_chars(word):
  return re.sub(r'(.)\1{2}', 'r\1', word)

def preprocess_text(text):
  if not isinstance(text, str):
    return [], ""

  # Lowercase
  text = text.lower()

  # Remove URL's and Email's
  text = re.sub(r'http\S+|www\S+|https\S+', '', text)
  text = re.sub(r'\S+@\S+', '', text)

  # Remove Numbers
  text = re.sub(r'\d+', '', text)

  # Remove Punctuation
  text = re.sub(r'[^\w\s]', '', text)

  # Remove Extra Spaces
  text = re.sub(r'\s+', ' ', text).strip()

  # Tokenize
  tokens = text.split()

  # Normalize Repeated Characters
  tokens = [normalize_repeated_chars(word) for word in tokens]

  # Remove short tokens except exceptions
  exceptions = ['no','not']
  tokens = [word for word in tokens if len(word) > 1 or word in exceptions]

  # clean_sentence
  clean_sentence = " ".join(tokens)

  return tokens,clean_sentence

## Task 3: Stress Testing

In [2]:
Test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"]

for text in Test_sentences:
  tokens, clean = preprocess_text(text)
  print("Original:",text)
  print("Tokens:", tokens)
  print("Cleaned:", clean)
  print('-'*50)

Original: Get 100% FREE access now!!!
Tokens: ['get', 'free', 'access', 'now']
Cleaned: get free access now
--------------------------------------------------
Original: I absolutely looooved this product 😍😍
Tokens: ['absolutely', 'lr\x01oved', 'this', 'product']
Cleaned: absolutely lroved this product
--------------------------------------------------
Original: Worst service ever... 0/10
Tokens: ['worst', 'service', 'ever']
Cleaned: worst service ever
--------------------------------------------------
Original: Call me at 9876543210
Tokens: ['call', 'me', 'at']
Cleaned: call me at
--------------------------------------------------
Original: This is THE best course!!!
Tokens: ['this', 'is', 'the', 'best', 'course']
Cleaned: this is the best course
--------------------------------------------------
Original: Visit https://openai.com now!
Tokens: ['visit', 'now']
Cleaned: visit now
--------------------------------------------------
Original: Nooooo this is baaad!!!
Tokens: ['nr\x01oo', '

## Task 4: Token Analytics

In [3]:
def token_analysis(tokens):
  if not tokens:
    return 0,0,0

  total = len(tokens)
  unique = len(set(tokens))
  average = sum(len(t) for t in tokens) / unique

  return total, unique, round(average, 2)

for text in Test_sentences:
  tokens, _ = preprocess_text(text)
  total, unique, average = token_analysis(tokens)
  print("Original:",text)
  print("Tokens:", tokens)
  print("Total:", total)
  print("Unique:", unique)
  print("Average:", average)
  print('-'*50)

Original: Get 100% FREE access now!!!
Tokens: ['get', 'free', 'access', 'now']
Total: 4
Unique: 4
Average: 4.0
--------------------------------------------------
Original: I absolutely looooved this product 😍😍
Tokens: ['absolutely', 'lr\x01oved', 'this', 'product']
Total: 4
Unique: 4
Average: 7.0
--------------------------------------------------
Original: Worst service ever... 0/10
Tokens: ['worst', 'service', 'ever']
Total: 3
Unique: 3
Average: 5.33
--------------------------------------------------
Original: Call me at 9876543210
Tokens: ['call', 'me', 'at']
Total: 3
Unique: 3
Average: 2.67
--------------------------------------------------
Original: This is THE best course!!!
Tokens: ['this', 'is', 'the', 'best', 'course']
Total: 5
Unique: 5
Average: 3.8
--------------------------------------------------
Original: Visit https://openai.com now!
Tokens: ['visit', 'now']
Total: 2
Unique: 2
Average: 4.0
--------------------------------------------------
Original: Nooooo this is baaad!!

### Analysis Questions
1. Which sentence had the most noise?
- Ans: **Most Noise** ---> `Win $$$ now!!! Limited offer!!!`
    - This sentence has heavy noise because of heavy symbols and spam pattern.

2. Which sentence retained the most meaningful tokens after cleaning?
- Ans: Most meaningful tokens: `"I absolutely looooved this product 😍😍"`
    - This sentence retains sentiment-heavy words like “absolutely”, “loved” and “product”

## Task 5: Frequency Analysis


In [4]:
from collections import Counter
all_tokens = []

for text in Test_sentences:
  tokens, _ = preprocess_text(text)
  all_tokens.extend(tokens)

freq = Counter(all_tokens)

print("Top 10 frequent Words:", freq.most_common(10))
print("Least 5 frequent Words:", freq.most_common()[-5:])

Top 10 frequent Words: [('this', 4), ('now', 3), ('ok', 3), ('is', 2), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('lr\x01oved', 1), ('product', 1)]
Least 5 frequent Words: [('offer', 1), ('am', 1), ('not', 1), ('happy', 1), ('with', 1)]


## Task 6: Build Full Pipeline

In [5]:
def full_pipeline(text_list):
  all_tokens = []
  clean_sentence = []

  for text in text_list:
    tokens, clean = preprocess_text(text)
    all_tokens.extend(tokens)
    clean_sentence.append(clean)

  freq = Counter(all_tokens)

  return {
      "tokens" : all_tokens,
      "clean_sentence" : clean_sentence,
  }

result = full_pipeline(Test_sentences)
print("Tokens:", result.get("tokens"))
print()
print("Cleaned_sentences:", result.get("clean_sentence"))

Tokens: ['get', 'free', 'access', 'now', 'absolutely', 'lr\x01oved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'me', 'at', 'this', 'is', 'the', 'best', 'course', 'visit', 'now', 'nr\x01oo', 'this', 'is', 'br\x01d', 'ok', 'ok', 'ok', 'got', 'it', 'win', 'now', 'limited', 'offer', 'am', 'not', 'happy', 'with', 'this']

Cleaned_sentences: ['get free access now', 'absolutely lr\x01oved this product', 'worst service ever', 'call me at', 'this is the best course', 'visit now', 'nr\x01oo this is br\x01d', 'ok ok ok got it', 'win now limited offer', 'am not happy with this']


## Task 7: Error Handling

In [6]:
edge_cases = ["", "😂😂😂", "123456789"]

for text in edge_cases:
  tokens, clean = preprocess_text(text)
  print("Original:",text)
  print("Tokens:", tokens)
  print("Cleaned:", clean)
  print('-'*50)

Original: 
Tokens: []
Cleaned: 
--------------------------------------------------
Original: 😂😂😂
Tokens: []
Cleaned: 
--------------------------------------------------
Original: 123456789
Tokens: []
Cleaned: 
--------------------------------------------------
